# Truncation Study — Stages 2 and 3

This notebook completes the experimental programme. It sweeps the truncation
point across its full grid for all three models and answers the two remaining
research questions.

**Stage 2 — preset against learned.** The proposed model and the Liquid State
Machine share a substrate exactly: the same neurons, the same connectivity, the
same initialization, the same neuron parameters. They differ in one respect only,
whether the reservoir is adapted by truncated surrogate-gradient descent, so the
difference between their curves measures what a short window of learning
recovers from a mathematically cheap initialization.

**Stage 3 — edge viability.** The proposed model is compared against a dense
LSTM of matched trainable capacity on classification quality, operation counts
and estimated energy.

The truncation point is swept rather than selected. A single best $k$ would
conceal the shape of the curve, and that shape is the result. Critically, the
LSTM is swept over the identical grid: a flat accuracy-versus-$k$ curve for the
spiking model alone would admit two explanations, and only sweeping the dense
baseline as well separates *the architecture tolerates truncation* from *the data
permits early classification*. The model-free prefix curves from Notebook 02
provide the third reference point.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from edgesnn.data_utils import make_datasets
from edgesnn.experiments import k_values, load_config, run_grid, seeds_for
from edgesnn.plotting import plot_confusion_matrix, plot_prefix_curve
from edgesnn.temporal_analysis import prefix_classifier_curve

DATA_DIR = "../Data"
FIG_DIR = "../reports/figures"
DATASETS = ["ECG5000", "Wafer"]
N_SEEDS = 5

# The cheap initializer carried forward from Stage 1. Set this from the
# `vs_gaussian` column of Notebook 04 rather than assuming it.
CHEAP_INIT = "sobol"

splits = {name: make_datasets(DATA_DIR, name) for name in DATASETS}
for name, (train_ds, _) in splits.items():
    print(f"{name:<12} T={train_ds.seq_len:>4}  k grid = {k_values(train_ds.seq_len)}")

## 1. The sweep

Four conditions are run at every truncation point: the proposed model under both
the Gaussian baseline and the selected cheap initializer, the LSM under the cheap
initializer, and the LSTM. The first two isolate the effect of the initializer
under training; the second and third form the Stage 2 comparison; the LSTM
supplies Stage 3.

Every condition uses the configuration frozen in Notebook 03. Nothing is re-tuned
per $k$ — doing so would fold tuning effects into the curve and make it
unreadable as a truncation result.

In [ ]:
sweep = {}

for name, (train_ds, test_ds) in splits.items():
    snn_cfg = load_config(f"config_snn_{name}")
    lsm_cfg = load_config(f"config_lsm_{name}")
    lstm_cfg = load_config(f"config_lstm_{name}")
    if snn_cfg is None or lsm_cfg is None or lstm_cfg is None:
        raise RuntimeError(f"Run notebook 03 first — configurations missing for {name}.")

    conditions = []
    for k in k_values(train_ds.seq_len):
        for method in ("gaussian", CHEAP_INIT):
            conditions += [dict(kind="snn", k=k, init_method=method, seed=s,
                                config=dict(snn_cfg))
                           for s in seeds_for(method, N_SEEDS)]
        conditions += [dict(kind="lsm", k=k, init_method=CHEAP_INIT, seed=s,
                            config=dict(lsm_cfg))
                       for s in seeds_for(CHEAP_INIT, N_SEEDS)]
        conditions += [dict(kind="lstm", k=k, seed=s, config=dict(lstm_cfg))
                       for s in range(N_SEEDS)]

    sweep[name] = run_grid(f"truncation_{name}", conditions, train_ds, test_ds,
                           max_epochs=80, min_epochs=15, patience=12)

In [ ]:
def curves_by_condition(frame, seq_len):
    """Group a sweep into one curve per (model, initializer), averaged over seeds."""
    frame = frame.copy()
    frame["condition"] = frame.apply(
        lambda r: r.model.upper() if r.model == "lstm"
        else f"{r.model.upper()} · {r.init_method}", axis=1)
    out = {}
    for label, group in frame.groupby("condition"):
        curve = (group.groupby("k")
                 .agg(macro_f1_mean=("macro_f1", "mean"),
                      macro_f1_std=("macro_f1", "std"),
                      energy_pj=("energy_pj", "mean"),
                      sops=("sops", "mean"), macs=("macs", "mean"))
                 .reset_index())
        curve["macro_f1_std"] = curve.macro_f1_std.fillna(0.0)
        curve["fraction"] = curve.k / seq_len
        out[label] = curve
    return out

model_curves = {name: curves_by_condition(frame, splits[name][0].seq_len)
                for name, frame in sweep.items()}
for name, curves in model_curves.items():
    print(f"{name}: {', '.join(curves)}")

## 2. Accuracy against the truncation point

Each panel overlays the model curves on the model-free prefix-classifier curve
recomputed from Notebook 02. The probe carries no recurrence and no temporal
inductive bias, so it is a floor rather than a competitor: a model tracking it
closely has demonstrated nothing that a linear classifier on raw amplitudes could
not, whereas the margin above it is what the architecture has earned.

In [ ]:
for name, (train_ds, _) in splits.items():
    probe = prefix_classifier_curve(train_ds.x, train_ds.y, seed=0)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    plot_prefix_curve(model_curves[name], x="fraction", ax=ax,
                      title=f"Accuracy vs. truncation point — {name}",
                      ylabel="Macro-F1 (test split)", legend_title="Condition")
    ax.plot(probe.fraction, probe.macro_f1_mean, linestyle="--", color="grey",
            marker="s", markersize=4, linewidth=1.6,
            label="linear prefix probe (train CV)")
    ax.legend(title="Condition", fontsize=9)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/{name}_truncation_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

## 3. Stage 2 — how much does training recover?

The proposed model and the LSM are isolated here under the same cheap
initializer. Their difference at each $k$ is the contribution of truncated
surrogate-gradient learning, with every other factor held identical by
construction.

In [ ]:
for name, curves in model_curves.items():
    trained = curves[f"SNN · {CHEAP_INIT}"].set_index("k")
    frozen = curves[f"LSM · {CHEAP_INIT}"].set_index("k")
    stage2 = pd.DataFrame({
        "fraction": trained.fraction,
        "trained (SNN)": trained.macro_f1_mean,
        "frozen (LSM)": frozen.macro_f1_mean,
        "recovered by training": trained.macro_f1_mean - frozen.macro_f1_mean,
        "energy ratio": trained.energy_pj / frozen.energy_pj,
    })
    print(f"\n{name} — Stage 2: effect of training the reservoir")
    display(stage2.round(4))

The energy ratio is reported beside the accuracy difference because the two
models run on the same substrate: any accuracy that training buys is bought at
whatever change it produces in the firing rate, and a ratio near unity means the
improvement was effectively free at inference time.

## 4. Stage 3 — accuracy against energy

The efficiency claim is a trade-off, so it is plotted as one. Each point is a
model at one truncation point; the horizontal axis is estimated arithmetic energy
per inference on a logarithmic scale, the vertical axis Macro-F1. A model lying
up and to the left dominates one lying down and to the right.

In [ ]:
for name, curves in model_curves.items():
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for label, curve in curves.items():
        ax.plot(curve.energy_pj / 1e6, curve.macro_f1_mean, marker="o", label=label)
        for _, row in curve.iterrows():
            ax.annotate(f"{row.fraction:.0%}", (row.energy_pj / 1e6, row.macro_f1_mean),
                        textcoords="offset points", xytext=(4, -9), fontsize=7.5,
                        color="grey")
    ax.set_xscale("log")
    ax.set_xlabel("Estimated arithmetic energy per inference (µJ, log scale)")
    ax.set_ylabel("Macro-F1 (test split)")
    ax.set_title(f"Accuracy against energy — {name}", fontsize=14, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/{name}_energy_tradeoff.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
headline = []
for name, curves in model_curves.items():
    lstm_full = curves["LSTM"].sort_values("k").iloc[-1]
    for label, curve in curves.items():
        best = curve.loc[curve.macro_f1_mean.idxmax()]
        headline.append({
            "dataset": name, "condition": label,
            "best k": int(best.k), "of T": f"{best.fraction:.0%}",
            "macro_f1": best.macro_f1_mean,
            "energy_uJ": best.energy_pj / 1e6,
            "energy vs full LSTM": lstm_full.energy_pj / best.energy_pj,
            "F1 vs full LSTM": best.macro_f1_mean - lstm_full.macro_f1_mean,
        })

headline = pd.DataFrame(headline)
display(headline.round(4))
headline.to_csv("../results/headline_summary.csv", index=False)

## 5. Error structure at the operating point

Scalar metrics say how much accuracy a truncated model gives up but not *where*.
A model that degrades by collapsing one class into another is a qualitatively
different outcome from one that degrades uniformly, and on an imbalanced
fault-detection task the distinction decides whether the model is usable at all.

In [ ]:
from edgesnn.evaluation_metrics import evaluate_efficiency
from edgesnn.experiments import build_model
from edgesnn.training import train_model

OPERATING_FRACTION = 0.4      # the cutoff to characterise; read it off Section 2

for name, (train_ds, test_ds) in splits.items():
    k = k_values(train_ds.seq_len)[int(OPERATING_FRACTION * 5) - 1]
    config = dict(load_config(f"config_snn_{name}"))
    lr = config.pop("lr")

    model = build_model("snn", num_classes=train_ds.num_classes, k=k,
                        init_method=CHEAP_INIT, seed=0, **config)
    train_model(model, train_ds, lr=lr, max_epochs=80, min_epochs=15,
                patience=12, seed=0, progress=False)
    metrics = evaluate_efficiency(model, test_ds, k=k)

    print(f"{name}: k={k} ({k / train_ds.seq_len:.0%}), "
          f"Macro-F1 {metrics['macro_f1']:.4f}, accuracy {metrics['accuracy']:.4f} "
          f"(majority floor {metrics['majority_rate']:.4f})")
    plot_confusion_matrix(
        metrics["confusion_matrix"], class_names=train_ds.class_names,
        title=f"{name} — SNN · {CHEAP_INIT} at k={k}",
        save_path=f"{FIG_DIR}/{name}_confusion_k{k}.png")

## 6. Reading the results

Three comparisons are available in the curves above, and they answer different
questions.

The **gap between the SNN and the LSM** at a given cutoff is what truncated
surrogate-gradient training contributes on an identical substrate. Because the
two share every architectural parameter, the difference cannot be attributed to
anything else, and if it comes at an unchanged energy cost then the training is
free at inference time.

The **gap between the SNN and the LSTM** is the accuracy price of the efficiency,
and it must be read against the energy axis rather than alone. The relevant
question is not whether the spiking model matches the dense one, but what it
costs to run at whatever accuracy it reaches.

The **gap between any model and the linear prefix probe** determines how much of
the truncation result belongs to the method at all. Where the probe already
saturates, the dataset permits early classification and no architecture may claim
credit for it; where the probe stays low, the margin above it was earned. This is
why the model-free analysis was run before any network was trained: it is the
only control that can distinguish an achievement of the method from a property of
the data.